# 証券営業インテリジェンス ハンズオン
## Part 3: ファンド目論見書の解析と検索インデックス構築

このノートブックでは、**野村アセットマネジメントの目論見書（PDF）** を Snowflake で
AI 解析して Cortex Search Service を構築します。

### 処理の流れ

```
Snowflake 内部ステージ（@PROSPECTUS_STAGE）← setup.sql でPDFコピー済み
      │ AI_PARSE_DOCUMENT
      ↓
RAW_FUND_DOCS テーブル（Markdownテキスト）
      │ SPLIT_TEXT_MARKDOWN_HEADER
      ↓
FUND_DOC_CHUNKS テーブル（セクション単位のチャンク）
      │ CREATE CORTEX SEARCH SERVICE
      ↓
FUND_DOCS_SEARCH_SERVICE（セマンティック検索）
```

### 前提条件
- `setup.sql` 実行済み（内部ステージ作成 & PDFコピー済み）
- `part1_security.ipynb` 実行済み

> ⏱️ **このパートの目安時間: 25分**

In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE COMPUTE_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 内部ステージのファイル確認

`setup.sql` で GitHub リポジトリから **PROSPECTUS_STAGE** にコピー済みの目論見書 PDF を確認します。

> ⏱️ **目安: 1分**

In [ ]:
-- setup.sql でコピー済みの目論見書PDFを確認
LIST @SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE;

## 3. AI_PARSE_DOCUMENT で PDF を解析

**AI_PARSE_DOCUMENT** は、PDF などのドキュメントを読み取り、
**Markdown 形式のテキスト**に変換します。

### 変換のイメージ

```
PDF（目論見書）
 ┌──────────────────────┐
 │ 1. ファンドの特色      │       AI_PARSE_DOCUMENT
 │   ・高配当株に投資     │  ─────────────────────→  # 1. ファンドの特色
 │ 2. 投資リスク          │                           ・高配当株に投資
 │   ・市場リスク         │                           # 2. 投資リスク
 │   ・為替リスク         │                           ## 市場リスク ...
 └──────────────────────┘
```

見出し・表・リストなどの構造が Markdown に変換されるため、
後続の **チャンク分割** がセクション単位で正確に行えます。

> ⏱️ **目安: 5〜10分**（PDF 1枚あたり数秒、ファイル数 × 並列処理）

In [ ]:
-- まず1ファイルだけ試してみる（動作確認）
SELECT
    RELATIVE_PATH                    AS "ファイル名",
    SIZE                             AS "ファイルサイズ",
    LEFT(
        PARSE_JSON(
            AI_PARSE_DOCUMENT(
                TO_FILE('@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE', RELATIVE_PATH),
                {'mode': 'LAYOUT'}
            )::VARCHAR
        )['content']::VARCHAR,
        500
    ) AS "解析結果（先頭500字）"
FROM DIRECTORY(@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE)
WHERE RELATIVE_PATH LIKE '%.pdf'
LIMIT 1;

In [ ]:
-- 全PDFを一括解析してテーブルに保存 ()
CREATE OR REPLACE TABLE RAW_FUND_DOCS AS
SELECT
    RELATIVE_PATH                AS FILE_NAME,
    SIZE                         AS FILE_SIZE_BYTES,
    CURRENT_TIMESTAMP()          AS PARSED_AT,
    PARSE_JSON(
        AI_PARSE_DOCUMENT(
            TO_FILE('@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        )::VARCHAR
    )['content']::VARCHAR AS PARSED_MARKDOWN
FROM DIRECTORY(@SNOWFINANCE_DB.DEMO_SCHEMA.PROSPECTUS_STAGE)
WHERE RELATIVE_PATH LIKE '%.pdf';

SELECT
    FILE_NAME       AS "ファイル名",
    FILE_SIZE_BYTES AS "サイズ",
    LENGTH(PARSED_MARKDOWN) AS "解析後文字数",
    PARSED_AT       AS "解析日時"
FROM RAW_FUND_DOCS
ORDER BY FILE_NAME;

## 4. セクション単位にチャンク分割

AI_PARSE_DOCUMENT で得た Markdown テキストを、
**SPLIT_TEXT_MARKDOWN_HEADER** でセクション（見出し）単位に分割します。

### なぜチャンク分割が必要か？

Cortex Search は**テキスト単位でインデックスを作成**します。
目論見書全体（数万字）を 1 チャンクにすると検索精度が下がります。
→ 「2. 投資リスク」「3. 費用」などのセクション単位に分割することで、
  質問に対して**最も関連するセクションだけ**を返せるようになります。

| 分割前 | 分割後 |
|---|---|
| 1ファイル = 1チャンク（数万字） | 1ファイル = 10〜30チャンク（数百字ずつ） |
| 検索精度が低い | セクション単位で精度高く検索 |

> ⏱️ **目安: 2分**

In [ ]:
-- SPLIT_TEXT_RECURSIVE_CHARACTER で均一サイズにチャンク分割
CREATE OR REPLACE TABLE FUND_DOC_CHUNKS AS
SELECT
    r.FILE_NAME,
    CASE r.FILE_NAME
        WHEN 'nomura_index_fund_topix_prospectus.pdf'    THEN '野村インデックスファンド・TOPIX'
        WHEN 'nomura_global_reit_monthly_prospectus.pdf' THEN '野村世界不動産投信(毎月分配型)'
        WHEN 'nomura_wrap_fund_standard_prospectus.pdf'  THEN 'のむラップ・ファンド(普通型)'
        ELSE REPLACE(r.FILE_NAME, '.pdf', '')
    END AS FUND_NAME,
    c.value::VARCHAR               AS CHUNK_CONTENT,
    LENGTH(c.value::VARCHAR)       AS CHUNK_LENGTH,
    c.index                        AS CHUNK_INDEX
FROM RAW_FUND_DOCS r,
LATERAL FLATTEN(
    SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
        r.PARSED_MARKDOWN,
        'markdown',
        1000,
        100
    )
) c
WHERE LENGTH(c.value::VARCHAR) > 50;

SELECT
    FUND_NAME         AS "ファンド名",
    COUNT(*)          AS "チャンク数",
    ROUND(AVG(CHUNK_LENGTH)) AS "平均文字数",
    MIN(CHUNK_LENGTH) AS "最小文字数",
    MAX(CHUNK_LENGTH) AS "最大文字数"
FROM FUND_DOC_CHUNKS
GROUP BY FUND_NAME
ORDER BY FUND_NAME;

In [ ]:
-- チャンクの中身を確認（各ファンドの最初の3チャンク）
SELECT
    FUND_NAME AS "ファンド名",
    CHUNK_INDEX AS "順番",
    CHUNK_LENGTH AS "文字数",
    LEFT(CHUNK_CONTENT, 200) AS "内容（先頭200字）"
FROM FUND_DOC_CHUNKS
WHERE CHUNK_INDEX <= 2
ORDER BY FUND_NAME, CHUNK_INDEX;

## 5. Cortex Search Service の作成

チャンクテーブルを元に、**FUND_DOCS_SEARCH_SERVICE** を作成します。

`FUND_NAME` と `SECTION_HEADER` を `ATTRIBUTES`（フィルタ可能な属性）として設定することで、
特定のファンドだけを絞り込んだ検索ができます。

> ⏱️ **目安: 3分**（インデックス構築は並行処理）

In [ ]:
-- Cortex Search Service: ファンド目論見書
CREATE OR REPLACE CORTEX SEARCH SERVICE FUND_DOCS_SEARCH_SERVICE
ON CHUNK_CONTENT
ATTRIBUTES FUND_NAME
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 day'
AS (
    SELECT
        FILE_NAME,
        FUND_NAME,
        CHUNK_INDEX,
        CHUNK_CONTENT
    FROM FUND_DOC_CHUNKS
);

In [ ]:
%%sql -r result_show_service
-- 作成した Cortex Search Service の確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## 6. セマンティック検索のデモ

### キーワード検索 vs セマンティック検索（アハ体験！）

目論見書の検索でも、Part 3 と同様に**意味を理解した検索**が威力を発揮します。

> **例**: 「株価が下がったときに損をするリスク」と検索
> → キーワード検索: 「市場リスク」「価格変動リスク」という見出しがなければヒットしない
> → Cortex Search: 意味を理解して「価格変動リスク」「市場リスク」のセクションを発見！

> ⏱️ **目安: 10分**

In [ ]:
-- 検索1: 「投資リスクが低い安定的なファンドを探したい」
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 300)               AS "内容（先頭300字）"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "投資リスクが低い安定的な運用ができるファンドの特徴",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- 検索2: 「信託報酬（手数料）の比較」
-- 複数ファンドの費用セクションを横断検索
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 300)               AS "費用の説明"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "信託報酬 手数料 費用 コスト",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 6
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f
ORDER BY 2, 1 DESC;

In [ ]:
-- 検索3: 「為替変動リスクについて知りたい」
-- 特定のリスク要因に関する説明を目論見書から抽出
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    f.value:FUND_NAME::STRING                              AS "ファンド名",
    LEFT(f.value:CHUNK_CONTENT::STRING, 400)               AS "説明内容"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "為替変動リスク 外貨建て資産 円高 円安の影響",
            "columns": ["FUND_NAME", "CHUNK_CONTENT"],
            "limit": 4
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- 検索4: 特定ファンドのみにフィルタして検索
-- 例: のむラップ・ファンド(普通型) の費用・コスト情報だけを取得
SELECT
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS "類似度スコア",
    LEFT(f.value:CHUNK_CONTENT::STRING, 400)               AS "内容"
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE',
        '{
            "query": "信託報酬や購入時手数料などのコスト・費用について",
            "columns": ["CHUNK_CONTENT"],
            "filter": {"@eq": {"FUND_NAME": "のむラップ・ファンド(普通型)"}},
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 7. Cortex Agent への組み込み

作成した `FUND_DOCS_SEARCH_SERVICE` を Cortex Agent のツールに追加することで、
**Snowflake Intelligence から目論見書に自然言語で質問**できるようになります。

### part6_cortex_agent.ipynb で追加するツール設定

`part6_cortex_agent.ipynb` の CREATE AGENT 文に以下を追加：

```json
{
    "tool_spec": {
        "type": "cortex_search",
        "name": "FUND_DOCS_SEARCH",
        "description": "野村AMファンドの目論見書を検索。ファンドの特色・投資リスク・信託報酬・分配金方針などを取得する"
    }
}
```

```json
"FUND_DOCS_SEARCH": {
    "name": "SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE",
    "max_results": 5,
    "title_column": "SECTION_HEADER",
    "id_column": "FILE_NAME"
}
```

### 追加後のデモ質問例

```
山田様（C001）のポートフォリオに含まれる資産クラスに近い
野村AMのファンドを目論見書の内容を踏まえて提案してください

教育資金贈与信託と野村のファンドを組み合わせた場合のメリットを教えてください

為替ヘッジなしのファンドにはどのようなリスクがありますか？
目論見書から具体的に教えてください
```

> **💡 ポイント**: 目論見書の情報と顧客データ・ニュースを組み合わせた回答が可能になります。

In [ ]:
-- データ確認: 作成したオブジェクトのサマリー
SELECT 'RAW_FUND_DOCS' AS "テーブル", COUNT(*) AS "件数",
       '解析済みPDFファイル数' AS "説明" FROM RAW_FUND_DOCS
UNION ALL
SELECT 'FUND_DOC_CHUNKS', COUNT(*), '検索インデックス用チャンク数' FROM FUND_DOC_CHUNKS
ORDER BY 1;

## まとめ

Part 3 完了！ファンド目論見書の AI 解析と検索インデックス構築が完了しました。

### 作成したオブジェクト

| オブジェクト | 種別 | 役割 |
|---|---|---|
| `FUND_DOCS_STAGE` | 内部ステージ | PDF を格納 |
| `RAW_FUND_DOCS` | テーブル | AI_PARSE_DOCUMENT の解析結果（Markdown） |
| `FUND_DOC_CHUNKS` | テーブル | セクション単位のチャンク |
| `FUND_DOCS_SEARCH_SERVICE` | Cortex Search | 目論見書のセマンティック検索 |

### このパートで使った機能

| 機能 | 役割 |
|---|---|
| `AI_PARSE_DOCUMENT` | PDFをMarkdownに変換（構造を保持） |
| `SPLIT_TEXT_MARKDOWN_HEADER` | 見出し単位にテキストをチャンク分割 |
| `CREATE CORTEX SEARCH SERVICE` | チャンクにセマンティック検索インデックスを作成 |

### ビジネス価値

- **今まで**: 目論見書を手で開いて読む → ファンドごとに数ページを読む
- **これから**: 「為替リスクが低いファンドは？」と聞くだけ → 複数ファンドを横断比較

**次のステップ:** `part4_cortex_analyst.ipynb` に進んでください。
または `part6_cortex_agent.ipynb` を更新して FUND_DOCS_SEARCH ツールを追加してください。